In [2]:
import pandas as pd
import re
original_df = pd.read_json("synthetic_incidents.jsonl", lines=True)

original_df.head()

,id,created_at,scenario_id,run_id,namespace,workload_name,image,real_pod_name,container_name,log_container_name,manifest_path,pod_describe,pod_logs,pod_logs_previous,deployment_yaml,apply_result
0,0f6ec139-c95d-429b-921c-dbfc98e82df4,2026-03-31 06:37:40.393570+00:00,pvc_not_found_mountfail,pn3zirfh,data-pipeline,busybox-pn3zirfh,busybox:1.36,busybox-pn3zirfh-86bbb7957c-7754l,app,app,k8s_out/pvc_not_found_mountfail-busybox-pn3zir...,Name: busybox-pn3zirfh-86bbb7957c-...,,,apiVersion: apps/v1\nkind: Deployment\nmetadat...,{'cmd': 'kubectl apply -f k8s_out/pvc_not_foun...
1,70607b4c-0478-4991-b8ff-c8e04c86b657,2026-03-31 06:37:45.527867+00:00,rbac_forbidden,hnw5520p,monitoring,kube-state-metrics-hnw5520p,registry.k8s.io/kube-state-metrics/kube-state-...,kube-state-metrics-hnw5520p-7fbb7866df-8rsgh,app,app,k8s_out/rbac_forbidden-kube-state-metrics-hnw5...,Name: kube-state-metrics-hnw5520p-...,I0331 06:37:30.545502 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,apiVersion: apps/v1\nkind: Deployment\nmetadat...,{'cmd': 'kubectl apply -f k8s_out/rbac_forbidd...
2,f2406c58-17ea-4a01-8638-62d04ccf9c54,2026-03-31 06:37:45.528651+00:00,rbac_forbidden,4g3nkfre,qa-app,ingress-nginx-4g3nkfre,registry.k8s.io/ingress-nginx/controller:v1.13.3,ingress-nginx-4g3nkfre-656c6b79b6-vbbbn,app,app,k8s_out/rbac_forbidden-ingress-nginx-4g3nkfre-...,Name: ingress-nginx-4g3nkfre-656c6...,----------------------------------------------...,----------------------------------------------...,apiVersion: apps/v1\nkind: Deployment\nmetadat...,{'cmd': 'kubectl apply -f k8s_out/rbac_forbidd...
3,a0e7f5e0-be03-4570-9348-10eef4fe9d6a,2026-03-31 06:37:45.528796+00:00,rbac_forbidden,n5oblcdc,prod-app,ingress-nginx-n5oblcdc,registry.k8s.io/ingress-nginx/controller:v1.13.3,ingress-nginx-n5oblcdc-78d4488dfd-gnt5g,app,app,k8s_out/rbac_forbidden-ingress-nginx-n5oblcdc-...,Name: ingress-nginx-n5oblcdc-78d44...,----------------------------------------------...,----------------------------------------------...,apiVersion: apps/v1\nkind: Deployment\nmetadat...,{'cmd': 'kubectl apply -f k8s_out/rbac_forbidd...
4,5f138bb5-69f3-4154-a82a-9ef85c022a55,2026-03-31 06:37:45.528997+00:00,rbac_forbidden,yrnzw8kt,deployment,kube-state-metrics-yrnzw8kt,registry.k8s.io/kube-state-metrics/kube-state-...,kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4,app,app,k8s_out/rbac_forbidden-kube-state-metrics-yrnz...,Name: kube-state-metrics-yrnzw8kt-...,I0331 06:37:30.542740 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,apiVersion: apps/v1\nkind: Deployment\nmetadat...,{'cmd': 'kubectl apply -f k8s_out/rbac_forbidd...


In [3]:
df = original_df[["scenario_id", "pod_describe", "pod_logs", "pod_logs_previous"]].copy()

In [4]:
df.head()

,scenario_id,pod_describe,pod_logs,pod_logs_previous
0,pvc_not_found_mountfail,Name: busybox-pn3zirfh-86bbb7957c-...,,
1,rbac_forbidden,Name: kube-state-metrics-hnw5520p-...,I0331 06:37:30.545502 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...
2,rbac_forbidden,Name: ingress-nginx-4g3nkfre-656c6...,----------------------------------------------...,----------------------------------------------...
3,rbac_forbidden,Name: ingress-nginx-n5oblcdc-78d44...,----------------------------------------------...,----------------------------------------------...
4,rbac_forbidden,Name: kube-state-metrics-yrnzw8kt-...,I0331 06:37:30.542740 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...


In [5]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in {"nan", "none"}:
        return ""
    x = x.replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ")
    lines = [line.rstrip() for line in x.split("\n")]
    return "\n".join(lines).strip()

In [6]:
for col in ["pod_describe", "pod_logs", "pod_logs_previous"]:
    df[col] = df[col].apply(normalize_text)

In [7]:
def extract_first(pattern, text, flags=re.IGNORECASE | re.MULTILINE):
    if not text:
        return None
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else None

In [8]:
def extract_block(text, block_name):
    if not text:
        return None

    lines = text.split("\n")
    block = []
    in_block = False

    for line in lines:
        if not in_block:
            if re.match(rf"^{re.escape(block_name)}:\s*$", line.strip()):
                in_block = True
            continue

        if line and not line.startswith(" ") and re.match(r"^[A-Za-z0-9_.\-/ ]+:\s*", line):
            break

        block.append(line)

    out = "\n".join(block).strip()
    return out if out else None

In [9]:
def parse_events_block(events_text):
    if not events_text:
        return {
            "event_reason": None,
            "event_message": None,
            "all_event_reasons": [],
            "all_event_messages": []
        }

    reasons = []
    messages = []

    lines = [line for line in events_text.split("\n") if line.strip()]

    for line in lines:
        if "Reason" in line and "Message" in line:
            continue
        if line.strip().startswith("----"):
            continue

        parts = re.split(r"\s{2,}", line.strip())
        if len(parts) >= 5:
            reasons.append(parts[1].strip())
            messages.append(parts[-1].strip())

    return {
        "event_reason": reasons[0] if reasons else None,
        "event_message": messages[0] if messages else None,
        "all_event_reasons": reasons,
        "all_event_messages": messages
    }

In [10]:
def parse_describe(text):
    containers_block = extract_block(text, "Containers")
    volumes_block = extract_block(text, "Volumes")
    tolerations_block = extract_block(text, "Tolerations")
    events_block = extract_block(text, "Events")

    events = parse_events_block(events_block)

    return {
        "pod_name": extract_first(r"^Name:\s*(.+)$", text),
        "namespace": extract_first(r"^Namespace:\s*(.+)$", text),
        "service_account_name": extract_first(r"^Service Account:\s*(.+)$", text),
        "node": extract_first(r"^Node:\s*(.+)$", text),
        "pod_status": extract_first(r"^Status:\s*(.+)$", text),
        "pod_ip": extract_first(r"^IP:\s*(.+)$", text),
        "controlled_by": extract_first(r"^Controlled By:\s*(.+)$", text),
        "image": extract_first(r"^\s*Image:\s*(.+)$", containers_block or text),
        "container_state": extract_first(r"^\s*State:\s*(.+)$", containers_block or text),
        "last_state": extract_first(r"^\s*Last State:\s*(.+)$", containers_block or text),
        "ready": extract_first(r"^\s*Ready:\s*(.+)$", containers_block or text),
        "restart_count": extract_first(r"^\s*Restart Count:\s*(.+)$", containers_block or text),
        "node_selectors": extract_first(r"^Node-Selectors:\s*(.+)$", text),
        "qos_class": extract_first(r"^QoS Class:\s*(.+)$", text),
        "claim_name": extract_first(r"^\s*ClaimName:\s*(.+)$", volumes_block or ""),
        "describe_events_raw": events_block,
        "describe_containers_raw": containers_block,
        "describe_volumes_raw": volumes_block,
        "describe_tolerations_raw": tolerations_block,
        "event_reason": events["event_reason"],
        "event_message": events["event_message"],
        "all_event_reasons": events["all_event_reasons"],
        "all_event_messages": events["all_event_messages"],
    }

In [11]:
describe_parsed = df["pod_describe"].apply(parse_describe).apply(pd.Series)
df = pd.concat([df, describe_parsed], axis=1)

In [12]:
def build_evidence_text(row):
    parts = []

    if row.get("pod_describe"):
        parts.append("POD DESCRIBE:\n" + row["pod_describe"])

    if row.get("pod_logs"):
        parts.append("POD LOGS:\n" + row["pod_logs"])

    if row.get("pod_logs_previous"):
        parts.append("POD LOGS PREVIOUS:\n" + row["pod_logs_previous"])

    return "\n\n".join(parts).strip() if parts else None

In [13]:
df["evidence_text"] = df.apply(build_evidence_text, axis=1)

In [14]:
parsed_df = df[
    [
        "scenario_id",
        "namespace",
        "pod_name",
        "service_account_name",
        "node",
        "pod_status",
        "image",
        "container_state",
        "last_state",
        "ready",
        "restart_count",
        "node_selectors",
        "claim_name",
        "event_reason",
        "event_message",
        "evidence_text",
    ]
].copy()

In [15]:
parsed_df.head(20)

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,evidence_text
0,pvc_not_found_mountfail,data-pipeline,busybox-pn3zirfh-86bbb7957c-7754l,default,<none>,Pending,busybox:1.36,NaN,NaN,NaN,NaN,<none>,missing-pvc-pn3zirfh,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,POD DESCRIBE:\nName: busybox-pn3zi...
1,rbac_forbidden,monitoring,kube-state-metrics-hnw5520p-7fbb7866df-8rsgh,app-sa-hnw5520p,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,NaN,True,0,<none>,NaN,Scheduled,Successfully assigned monitoring/kube-state-me...,POD DESCRIBE:\nName: kube-state-me...
2,rbac_forbidden,qa-app,ingress-nginx-4g3nkfre-656c6b79b6-vbbbn,app-sa-4g3nkfre,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned qa-app/ingress-nginx-4g3...,POD DESCRIBE:\nName: ingress-nginx...
3,rbac_forbidden,prod-app,ingress-nginx-n5oblcdc-78d4488dfd-gnt5g,app-sa-n5oblcdc,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned prod-app/ingress-nginx-n...,POD DESCRIBE:\nName: ingress-nginx...
4,rbac_forbidden,deployment,kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4,app-sa-yrnzw8kt,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,NaN,True,0,<none>,NaN,Scheduled,Successfully assigned deployment/kube-state-me...,POD DESCRIBE:\nName: kube-state-me...
5,pvc_not_found_mountfail,app,redis-wuwdazqz-6cbddc486f-dt8vw,default,<none>,Pending,redis:7.2,NaN,NaN,NaN,NaN,<none>,missing-pvc-wuwdazqz,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,POD DESCRIBE:\nName: redis-wuwdazq...
6,pvc_not_found_mountfail,monitoring,nginx-0kbrd6uu-65c96d8888-5qkdp,default,<none>,Pending,nginx:1.27,NaN,NaN,NaN,NaN,<none>,missing-pvc-0kbrd6uu,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,POD DESCRIBE:\nName: nginx-0kbrd6u...
7,pvc_not_found_mountfail,app-deployment,nginx-6ggb2my5-9b59965bd-rbdvh,default,<none>,Pending,nginx:1.27,NaN,NaN,NaN,NaN,<none>,missing-pvc-6ggb2my5,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,POD DESCRIBE:\nName: nginx-6ggb2my...
8,pvc_not_found_mountfail,prod-app,nginx-99p30cd9-7cf76c86d4-qjkl7,default,<none>,Pending,nginx:1.27,NaN,NaN,NaN,NaN,<none>,missing-pvc-99p30cd9,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,POD DESCRIBE:\nName: nginx-99p30cd...
9,rbac_forbidden,qa-app,kube-state-metrics-a7leoiiu-545cbf487c-jp7hq,app-sa-a7leoiiu,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,NaN,True,0,<none>,NaN,Scheduled,Successfully assigned qa-app/kube-state-metric...,POD DESCRIBE:\nName: kube-state-me...


In [ ]:
# preview_df = sample_df.copy()

# for col in ["event_message", "evidence_text"]:
#     if col in preview_df.columns:
#         preview_df[col] = preview_df[col].fillna("").str.slice(0, 200)

# preview_df

NameError: name 'sample_df' is not defined

In [17]:
final_df = df.copy()

In [18]:
cols = [
    "scenario_id",
    "namespace",
    "pod_name",
    "service_account_name",
    "node",
    "pod_status",
    "image",
    "container_state",
    "last_state",
    "ready",
    "restart_count",
    "node_selectors",
    "claim_name",
    "event_reason",
    "event_message",
    "pod_describe",
    "pod_logs",
    "pod_logs_previous",
    "evidence_text",
]

In [19]:
final_df = df[cols].copy()
final_df.head()

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,pod_describe,pod_logs,pod_logs_previous,evidence_text
0,pvc_not_found_mountfail,data-pipeline,busybox-pn3zirfh-86bbb7957c-7754l,default,<none>,Pending,busybox:1.36,NaN,NaN,NaN,NaN,<none>,missing-pvc-pn3zirfh,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,Name: busybox-pn3zirfh-86bbb7957c-...,,,POD DESCRIBE:\nName: busybox-pn3zi...
1,rbac_forbidden,monitoring,kube-state-metrics-hnw5520p-7fbb7866df-8rsgh,app-sa-hnw5520p,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,NaN,True,0,<none>,NaN,Scheduled,Successfully assigned monitoring/kube-state-me...,Name: kube-state-metrics-hnw5520p-...,I0331 06:37:30.545502 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,POD DESCRIBE:\nName: kube-state-me...
2,rbac_forbidden,qa-app,ingress-nginx-4g3nkfre-656c6b79b6-vbbbn,app-sa-4g3nkfre,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned qa-app/ingress-nginx-4g3...,Name: ingress-nginx-4g3nkfre-656c6...,----------------------------------------------...,----------------------------------------------...,POD DESCRIBE:\nName: ingress-nginx...
3,rbac_forbidden,prod-app,ingress-nginx-n5oblcdc-78d4488dfd-gnt5g,app-sa-n5oblcdc,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1,<none>,NaN,Scheduled,Successfully assigned prod-app/ingress-nginx-n...,Name: ingress-nginx-n5oblcdc-78d44...,----------------------------------------------...,----------------------------------------------...,POD DESCRIBE:\nName: ingress-nginx...
4,rbac_forbidden,deployment,kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4,app-sa-yrnzw8kt,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,NaN,True,0,<none>,NaN,Scheduled,Successfully assigned deployment/kube-state-me...,Name: kube-state-metrics-yrnzw8kt-...,I0331 06:37:30.542740 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,POD DESCRIBE:\nName: kube-state-me...


In [20]:
final_df.to_json("final_df.jsonl", orient="records", lines=True)